# Benchmark de asociacion (PostgreSQL)

Este notebook ejecuta benchmark y sensibilidad de reglas de asociacion con filtros de calidad y consenso multicriterio.


In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

CWD = Path.cwd().resolve()
candidates = [
    CWD,
    CWD.parent,
    CWD / "03_modelado" / "proyecto_ml_experimentos",
]
ROOT = next((p for p in candidates if (p / "src" / "association.py").exists()), None)
if ROOT is None:
    raise RuntimeError("No se encontro la raiz de proyecto_ml_experimentos (src/association.py).")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for m in ["src", "src.datasets_postgres", "src.association"]:
    if m in sys.modules:
        del sys.modules[m]

from src.datasets_postgres import load_association_dataset
from src.association import benchmark_association, benchmark_association_sensitivity


In [ ]:
df = load_association_dataset()
display(df.head())

tx_size = df.groupby("transaccion_id")["producto"].nunique()

display(
    Markdown(
        f'''
### Calidad del dataset
- Filas: **{len(df):,}**
- Transacciones: **{df['transaccion_id'].nunique():,}**
- Productos unicos: **{df['producto'].nunique():,}**
- Transacciones con >=2 productos: **{int((tx_size >= 2).sum()):,}**
'''
    )
)


In [ ]:
res = benchmark_association(
    df,
    min_support=0.02,
    min_confidence=0.25,
    top_k=20,
    min_realized_conf_val=0.25,
    min_realized_conf_test=0.25,
    min_realized_support=0.005,
)

sens = benchmark_association_sensitivity(
    df,
    support_grid=(0.015, 0.02, 0.03),
    confidence_grid=(0.25, 0.30, 0.35),
    top_k=50,
    min_realized_conf_val=0.25,
    min_realized_conf_test=0.25,
    min_realized_support=0.005,
)

models_dir = ROOT / "models"
models_dir.mkdir(exist_ok=True)
charts_dir = ROOT.parents[1] / "05_evidencias" / "graficas"
charts_dir.mkdir(parents=True, exist_ok=True)
res.to_csv(models_dir / "benchmark_association.csv", index=False)
sens.to_csv(models_dir / "benchmark_association_sensitivity.csv", index=False)

display(res)
display(sens.head(15))


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(res["algoritmo"], res["score_general"])
plt.title("Score general por algoritmo")
plt.ylabel("score_general")
plt.tight_layout()
plt.savefig(charts_dir / "association_score_por_algoritmo.png", dpi=150)
plt.show()

for algo, sub in sens.groupby("algoritmo"):
    tmp = sub.sort_values(["min_support", "min_confidence"])
    plt.figure(figsize=(8, 4))
    plt.plot(tmp["min_support"].astype(str) + "|" + tmp["min_confidence"].astype(str), tmp["score_general"], marker="o")
    plt.title(f"Sensibilidad de score_general - {algo}")
    plt.ylabel("score_general")
    plt.xlabel("support|confidence")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(charts_dir / f"association_sensibilidad_{algo.lower()}.png", dpi=150)
    plt.show()

display(Markdown(f"Graficas guardadas en: `{charts_dir}`"))


In [ ]:
best = res.sort_values("score_general", ascending=False).iloc[0]
best_sens = sens.sort_values("score_general", ascending=False).iloc[0]

if float(best["conf_test_prom"]) >= 0.45 and float(best["jaccard_train_test"]) >= 0.8:
    semaforo = "VERDE"
    estado = "reglas estables y confiables para uso operativo"
elif float(best["conf_test_prom"]) >= 0.30 and float(best["jaccard_train_test"]) >= 0.5:
    semaforo = "AMARILLO"
    estado = "resultado util con necesidad de revisar reglas manualmente"
else:
    semaforo = "ROJO"
    estado = "calidad insuficiente para usar reglas sin ajustes"

display(
    Markdown(
        f'''
## Interpretacion
- Mejor algoritmo en este corte: **{best['algoritmo']}**.
- `score_general`: **{best['score_general']:.4f}**.
- Reglas train: **{int(best['reglas_train'])}**.
- Reglas que pasan filtro de calidad: **{int(best['reglas_filtradas_calidad'])}**.
- Top consenso usado para evaluacion: **{int(best['reglas_top_consenso'])}**.
- Configuracion de sensibilidad ganadora: support **{best_sens['min_support']:.3f}**, confidence **{best_sens['min_confidence']:.2f}**.

Lectura recomendada:
1. Mantener solo reglas que pasen umbrales de calidad para reducir ruido.
2. Priorizar estabilidad (Jaccard) y confianza en test sobre cantidad total de reglas.
3. Si Apriori y FPGrowth empatan en calidad, elegir por tiempo de ejecucion.

## Conclusion ejecutiva
- Semaforo: **{semaforo}**.
- Estado: **{estado}**.
'''
    )
)
